# Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

import openpyxl

# Load engineered dataset

In [ ]:
project_root = Path.cwd().parent
features_path = project_root / "data" / "spy_features.csv"

df = pd.read_csv(features_path)
df["Date"] = pd.to_datetime(df["Date"])

df.head()

# Define features and target

In [ ]:
feature_cols = [
    "return_1d",
    "return_2d",
    "return_5d",
    "return_10d",
    "return_20d",
    "ma_5_rel",
    "ma_10_rel",
    "ma_20_rel",
    "ma_spread_5_20",
    "ma_spread_10_20",
    "vol_5",
    "vol_20",
    "vol_ratio",
    "volume_change_1d",
    "range_pos_10",
    "range_pos_20",
    "intraday_return",
    "high_low_range",
    "close_to_high",
    "close_to_low"
]

X = df[feature_cols]
y = df["target"]

# Chronological train / validation / test split

In [ ]:
n = len(df)

train_end = int(0.7 * n)
val_end = int(0.85 * n)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

# Train logistic regression with feature scaling

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

val_pred = model.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, val_pred))

# Evaluate classification performance

In [ ]:
val_proba = model.predict_proba(X_val)[:, 1]

print("Validation accuracy:", accuracy_score(y_val, val_pred))
print("Validation AUC:", roc_auc_score(y_val, val_proba))
print("Fraction predicted up:", val_pred.mean())
print("Probability min/max/mean:", val_proba.min(), val_proba.max(), val_proba.mean())
print(classification_report(y_val, val_pred))
print(confusion_matrix(y_val, val_pred))

# Train a nonlinear baseline: random forest

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42
)

model.fit(X_train, y_train)

val_pred = model.predict(X_val)

# Evaluate classification performance

In [ ]:
val_proba = model.predict_proba(X_val)[:, 1]

print("Validation accuracy:", accuracy_score(y_val, val_pred))
print("Validation AUC:", roc_auc_score(y_val, val_proba))
print("Fraction predicted up:", val_pred.mean())
print("Probability min/max/mean:", val_proba.min(), val_proba.max(), val_proba.mean())
print(classification_report(y_val, val_pred))
print(confusion_matrix(y_val, val_pred))

## Expanding-window walk-forward evaluation

In [ ]:
n = len(df)
train_end = int(0.7 * n)

walk_preds = []
walk_probas = []
walk_true = []
walk_dates = []

for t in range(train_end, n):
    X_train = X.iloc[:t]
    y_train = y.iloc[:t]

    X_next = X.iloc[[t]]
    y_next = y.iloc[t]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000))
    ])

    model.fit(X_train, y_train)

    pred = model.predict(X_next)[0]
    proba = model.predict_proba(X_next)[0, 1]

    walk_preds.append(pred)
    walk_probas.append(proba)
    walk_true.append(y_next)
    walk_dates.append(df.iloc[t]["Date"])

## Evaluate classification performance

In [ ]:
print("Walk-forward accuracy:", accuracy_score(walk_true, walk_preds))
print("Walk-forward AUC:", roc_auc_score(walk_true, walk_probas))
print("Fraction predicted up:", sum(walk_preds) / len(walk_preds))
print(classification_report(walk_true, walk_preds))
print(confusion_matrix(walk_true, walk_preds))

## Better than naively always predicting up?

In [ ]:
print("Walk-forward true up fraction:", sum(walk_true) / len(walk_true))

always_up_preds = [1] * len(walk_true)
print("Always-up accuracy:", accuracy_score(walk_true, always_up_preds))

# Results

In [ ]:
walk_true_up_fraction = sum(walk_true) / len(walk_true)
always_up_preds = [1] * len(walk_true)
always_up_accuracy = accuracy_score(walk_true, always_up_preds)

results = pd.DataFrame([
    {
        "model": "logreg_scaled_fixed_split",
        "target_horizon_days": 1,
        "evaluation": "validation",
        "accuracy": 0.5000,
        "auc": 0.4929,
        "predicted_up_fraction": 0.9326
    },
    {
        "model": "random_forest_fixed_split",
        "target_horizon_days": 1,
        "evaluation": "validation",
        "accuracy": 0.5164,
        "auc": 0.4807,
        "predicted_up_fraction": 0.9391
    },
    {
        "model": "logreg_scaled_walk_forward",
        "target_horizon_days": 1,
        "evaluation": "walk_forward",
        "accuracy": accuracy_score(walk_true, walk_preds),
        "auc": roc_auc_score(walk_true, walk_probas),
        "predicted_up_fraction": sum(walk_preds) / len(walk_preds)
    },
    {
        "model": "always_up_walk_forward",
        "target_horizon_days": 1,
        "evaluation": "walk_forward",
        "accuracy": always_up_accuracy,
        "auc": np.nan,
        "predicted_up_fraction": 1.0
    }
])

results_csv_path = project_root / "results" / "model_comparison.csv"
results_xlsx_path = project_root / "results" / "model_comparison.xlsx"

results.to_csv(results_csv_path, index=False, sep=";")
results.to_excel(results_xlsx_path, index=False)

results